# 3) Daily Streak Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Build a daily study habit by recording a 1-line check-in and computing a streak.

## Simple rules

- One line per day (replace allowed).
- Save logs in memory.
- Compute current streak and longest streak.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "03_streak_memory.json"

import re
from datetime import datetime, timedelta

def today_key() -> str:
    return datetime.now().strftime("%Y-%m-%d")

def compute_streak(dates: List[str]) -> int:
    if not dates:
        return 0
    dayset = set(dates)
    streak = 0
    today = datetime.now().date()
    for i in range(0, 3650):
        d = (today - timedelta(days=i)).strftime("%Y-%m-%d")
        if d in dayset:
            streak += 1
        else:
            break
    return streak

def _parse_minutes(text: str) -> int:
    m = re.match(r'^\s*(\d+)\s*(?:min|mins|minutes)?\b', text.lower())
    if m:
        return int(m.group(1))
    return 0

def decide_next_action(user_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(user_text):
        return Action("terminate", {})

    entry = user_text.strip()
    if not entry:
        return Action("notify", {"message": "Write something (e.g., '10 mins arrays')."})

    cmd = entry.lower()
    logs = memory.get("logs", [])

    # Commands: list, weekly, export, edit:, delete:
    if cmd in {"list", "logs"}:
        if not logs:
            return Action("notify", {"message": "No logs yet."})
        lines = [f"{i+1}. {l['date']} - {l.get('minutes',0)}m - {l['text']}" for i, l in enumerate(logs[-14:])]
        return Action("notify", {"message": "Recent logs:\n" + "\n".join(lines)})

    if cmd in {"weekly", "summary"}:
        today = datetime.now().date()
        totals = []
        for i in range(6, -1, -1):
            d = (today - timedelta(days=i)).strftime("%Y-%m-%d")
            m = sum(int(l.get('minutes',0)) for l in logs if l.get('date') == d)
            totals.append((d, m))
        week_total = sum(m for _, m in totals)
        avg = week_total / 7.0
        lines = [f"{d}: {m} mins" for d, m in totals]
        msg = f"Last 7 days total: {week_total} mins (avg {avg:.1f} mins/day)\n" + "\n".join(lines)
        return Action("notify", {"message": msg})

    if cmd.startswith("export"):
        return Action("export_csv", {})

    if cmd.startswith("edit:"):
        payload = entry.split(":", 1)[1].strip() if ':' in entry else ''
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(logs):
                return Action("notify", {"message": "Use 'update:IDX | text' to change this entry."})
            return Action("notify", {"message": "Index out of range."})
        return Action("notify", {"message": "Invalid edit command."})

    if cmd.startswith("delete:"):
        payload = entry.split(":", 1)[1].strip() if ':' in entry else ''
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(logs):
                removed = logs.pop(idx)
                memory["logs"] = logs
                return Action("notify", {"message": f"Deleted: {removed['date']} - {removed.get('minutes',0)}m - {removed['text']}"})
            return Action("notify", {"message": "Index out of range."})
        return Action("notify", {"message": "Invalid delete command."})

    # Otherwise treat entry as a daily log, optional leading minutes
    mins = _parse_minutes(entry)
    d = today_key()
    # remove existing today's entry (replace allowed)
    logs = [x for x in logs if x.get('date') != d]
    logs.append({"date": d, "text": entry, "minutes": mins})
    memory["logs"] = logs

    dates = [x["date"] for x in logs]
    memory["streak"] = compute_streak(dates)
    memory["longest"] = max(int(memory.get("longest", 0)), int(memory["streak"]))

    # Reward milestones (7, 14, 30 days)
    milestones = {7: "Nice! 1-week streak reward unlocked.", 14: "Two-week streak! Reward unlocked.", 30: "30-day streak! Big reward unlocked."}
    msg = f"Saved for {d}. Streak: {memory['streak']}. Longest: {memory['longest']}."
    if memory['streak'] in milestones and memory.get("last_reward") != memory['streak']:
        msg += f"\n{milestones[memory['streak']] }"
        memory["last_reward"] = memory['streak']

    return Action("notify", {"message": msg})

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}

    Tools.notify("Daily Streak Agent is running.")
    Tools.notify("Type your 1-line study log (e.g., '10 mins arrays'). Type 'list', 'weekly', 'export', or 'stop'.")

    user_text = Tools.ask("What did you study today (1 line)?")
    while True:
        action = decide_next_action(user_text, memory)
        out = env.execute(action, memory)
        memory = out["memory"]

        if out["result"] == "terminated":
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved. Bye!")
            break

        # handle export result in Environment (writes CSV)
        if out["result"] == "exported_csv":
            Tools.notify("Exported logs to CSV.")

        env.execute(Action("save_memory", {}), memory)
        user_text = Tools.ask("Update today's line or command (or stop)")

run_agent()


## Easy extensions

- Save minutes spent (prefix your log with minutes: e.g., '10 mins arrays').
- Weekly summary (`weekly` or `summary`) showing last 7 days totals and average.
- Reward messages at milestones (7/14/30 day streaks).
- Export logs to CSV: use `export` to create `03_streak_memory.csv`.
- Edit or delete past entries using `edit:IDX` and `delete:IDX`.
- List recent logs with `list` or `logs`.

Usage examples:

- `10 mins arrays` -> logs 10 minutes for today with text '10 mins arrays'.
- `weekly` -> shows totals for the last 7 days.
- `export` -> writes CSV of logs.